# BERTopic Clustering

Clusters papers into topics and macro-categories using BERTopic.

**Outputs** (in `data/processed/`):
- `dataset_with_clusters.csv` - main dataset with topic_id, macro_id columns
- `topic_macro_mapping.csv` - lookup table: topic_id → topic_name, macro_id, macro_name

In [11]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
from umap import UMAP
from hdbscan import HDBSCAN

try:
    from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
    from bertopic.vectorizers import ClassTfidfTransformer
    HAS_EXTRAS = True
except: HAS_EXTRAS = False

In [12]:
# Config
INPUT = Path("../data/processed/dataset_clean.csv")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

NR_TOPICS = 30
N_MACRO = 12
MIN_TOPIC_SIZE = 15
MIN_DOC_LEN = 30

In [13]:
# Load data
df = pd.read_csv(INPUT)
print(f"Loaded: {len(df):,} papers")

# Build documents
def clean(x): 
    return "" if pd.isna(x) else re.sub(r"\s+", " ", str(x)).strip()

def build_doc(r):
    parts = [clean(r.get(c, "")) for c in ["Title", "Abstract", "AuthorKeywords"]]
    return " . ".join([p for p in parts if p])

docs = df.apply(build_doc, axis=1).tolist()

# Filter short docs
keep = [i for i, d in enumerate(docs) if len(d) >= MIN_DOC_LEN]
df = df.iloc[keep].reset_index(drop=True)
docs = [docs[i] for i in keep]
print(f"After filtering: {len(df):,} papers")

Loaded: 3,530 papers
After filtering: 3,530 papers


In [14]:
# Build BERTopic model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

domain_stop = {"visualization", "visualizations", "visual", "analytics", "analysis",
               "approach", "method", "technique", "system", "framework", "model",
               "data", "dataset", "interactive", "user", "paper", "results", "using"}
stop_words = list(set(ENGLISH_STOP_WORDS).union(domain_stop))

vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1,3), min_df=3, max_df=0.6)
umap_model = UMAP(n_neighbors=25, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=max(2, MIN_TOPIC_SIZE//3),
                        metric="euclidean", cluster_selection_method="eom", prediction_data=True)

ctfidf = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True) if HAS_EXTRAS else None
repr_model = [KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.4)] if HAS_EXTRAS else None

topic_model = BERTopic(
    embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer, ctfidf_model=ctfidf, representation_model=repr_model,
    min_topic_size=MIN_TOPIC_SIZE, calculate_probabilities=True, verbose=True
)

In [15]:
# Fit model
topics, probs = topic_model.fit_transform(docs)

# First reduce to target topics
if NR_TOPICS:
    topic_model.reduce_topics(docs, nr_topics=NR_TOPICS)
    topics = topic_model.topics_

# Then reduce outliers (after topic reduction to avoid warning)
try:
    topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, threshold=0.05)
    topic_model.update_topics(docs, topics=topics, vectorizer_model=vectorizer, representation_model=repr_model)
    print("Outlier reduction applied")
except Exception as e: 
    print(f"Outlier reduction skipped: {e}")

n_outliers = sum(1 for t in topics if t == -1)
print(f"Topics: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outliers: {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")

2026-01-09 21:56:32,295 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/111 [00:00<?, ?it/s]

2026-01-09 22:19:44,298 - BERTopic - Embedding - Completed ✓
2026-01-09 22:19:44,313 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-09 22:20:08,303 - BERTopic - Dimensionality - Completed ✓
2026-01-09 22:20:08,312 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-09 22:20:08,932 - BERTopic - Cluster - Completed ✓
2026-01-09 22:20:08,959 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-09 22:24:06,678 - BERTopic - Representation - Completed ✓
2026-01-09 22:24:07,709 - BERTopic - Topic reduction - Reducing number of topics
2026-01-09 22:24:07,741 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-09 22:26:10,568 - BERTopic - Representation - Completed ✓
2026-01-09 22:26:10,659 - BERTopic - Topic reduction - Reduced number of topics from 56 to 30
100%|██████████| 1/1 [00:02<00:00,  2.59s/it]
2026-01-09 22:26:14,100 - BERTopic - WARNING: Using a custom list of 

Outlier reduction applied
Topics: 29
Outliers: 27 (0.8%)


In [16]:
# Generate topic labels
topic_info = topic_model.get_topic_info()

try:
    labels = topic_model.generate_topic_labels(nr_words=4, topic_prefix=False, separator=" • ")
    label_map = dict(zip(topic_info["Topic"].tolist(), labels))
except:
    label_map = {t: " • ".join([w for w,_ in topic_model.get_topic(t)[:4]]) 
                 if t != -1 else "Other" for t in topic_info["Topic"]}

topic_info["TopicName"] = topic_info["Topic"].map(label_map)
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs,TopicName
0,-1,27,-1_graphical interfaces_automatic presentation...,"[graphical interfaces, automatic presentation,...",[From Information to Choice: A Critical Inquir...,graphical interfaces • automatic presentation ...
1,0,429,0_unstructured meshes_unstructured grids_direc...,"[unstructured meshes, unstructured grids, dire...",[Two-phase perspective ray casting for interac...,unstructured meshes • unstructured grids • dir...
2,1,249,1_dimensionality reduction_dimensional dataset...,"[dimensionality reduction, dimensional dataset...",[Value and Relation Display for Interactive Ex...,dimensionality reduction • dimensional dataset...
3,2,238,2_flow field_flow fields_vortices_computationa...,"[flow field, flow fields, vortices, computatio...",[Vortex Lens: Interactive Vortex Core Line Ext...,flow field • flow fields • vortices • computat...
4,3,218,3_node link diagrams_link diagrams_graph layou...,"[node link diagrams, link diagrams, graph layo...",[Divided Edge Bundling for Directional Network...,node link diagrams • link diagrams • graph lay...
5,4,196,4_infographics_narratives_storytelling_infogra...,"[infographics, narratives, storytelling, infog...",[Visualization Design Practices in a Crisis: B...,infographics • narratives • storytelling • inf...
6,5,176,5_dataflow_developers_prototyping_graphic,"[dataflow, developers, prototyping, graphic, j...",[Animated Vega-Lite: Unifying Animation with a...,dataflow • developers • prototyping • graphic
7,6,145,6_blood vessels_vascular_computed tomography_t...,"[blood vessels, vascular, computed tomography,...",[CPR - curved planar reformation . Visualizati...,blood vessels • vascular • computed tomography...
8,7,111,7_simulation ensembles_ensembles_ensemble_weat...,"[simulation ensembles, ensembles, ensemble, we...",[Multi-Resolution Climate Ensemble Parameter A...,simulation ensembles • ensembles • ensemble • ...
9,8,150,8_topic modeling_exploratory search_informatio...,"[topic modeling, exploratory search, informati...",[Large-Scale Evaluation of Topic Models and Di...,topic modeling • exploratory search • informat...


In [17]:
# Group into macro categories
topic_ids = topic_info.loc[topic_info.Topic != -1, "Topic"].astype(int).tolist()
topic_emb = normalize(topic_model.topic_embeddings_[topic_ids])

macro_ids = AgglomerativeClustering(n_clusters=N_MACRO, linkage='ward').fit_predict(topic_emb)
macro_map = dict(zip(topic_ids, macro_ids))

# Generate macro labels
def macro_label(mid):
    member_topics = [t for t in topic_ids if macro_map[t] == mid]
    c = Counter()
    for t in member_topics:
        for w, _ in topic_model.get_topic(t)[:10]: c[w] += 1
    return " • ".join([w for w, _ in c.most_common(4)])

macro_label_map = {m: macro_label(m) for m in sorted(set(macro_ids))}

# Add to topic_info
topic_info["MacroId"] = topic_info["Topic"].map(lambda t: macro_map.get(int(t), -1))
topic_info["MacroName"] = topic_info["MacroId"].map(lambda m: macro_label_map.get(m, "Other"))

print(f"Macro categories: {N_MACRO}")
topic_info[["Topic", "TopicName", "MacroId", "MacroName", "Count"]].head(15)

Macro categories: 12


,Topic,TopicName,MacroId,MacroName,Count
0,-1,graphical interfaces • automatic presentation ...,-1,Other,27
1,0,unstructured meshes • unstructured grids • dir...,2,direct volume rendering • infovis • augmented ...,429
2,1,dimensionality reduction • dimensional dataset...,0,dimensionality reduction • dimensional dataset...,249
3,2,flow field • flow fields • vortices • computat...,2,direct volume rendering • infovis • augmented ...,238
4,3,node link diagrams • link diagrams • graph lay...,1,node link diagrams • link diagrams • graph lay...,218
5,4,infographics • narratives • storytelling • inf...,2,direct volume rendering • infovis • augmented ...,196
6,5,dataflow • developers • prototyping • graphic,3,dataflow • developers • prototyping • graphic,176
7,6,blood vessels • vascular • computed tomography...,2,direct volume rendering • infovis • augmented ...,145
8,7,simulation ensembles • ensembles • ensemble • ...,0,dimensionality reduction • dimensional dataset...,111
9,8,topic modeling • exploratory search • informat...,1,node link diagrams • link diagrams • graph lay...,150


In [18]:
# Add clusters to dataset
doc_info = topic_model.get_document_info(docs, df=df)

df["topic_id"] = doc_info["Topic"].values
df["macro_id"] = df["topic_id"].map(lambda t: macro_map.get(int(t), -1) if pd.notna(t) else -1)

# Save dataset
df.to_csv(OUT / "dataset_with_clusters.csv", index=False)
print(f"✓ dataset_with_clusters.csv | {len(df):,} rows")

✓ dataset_with_clusters.csv | 3,530 rows


In [19]:
# Save topic-macro mapping
mapping = topic_info[["Topic", "TopicName", "MacroId", "MacroName", "Count"]].copy()
mapping.columns = ["topic_id", "topic_name", "macro_id", "macro_name", "count"]
mapping = mapping.sort_values(["macro_id", "count"], ascending=[True, False])
mapping.to_csv(OUT / "topic_macro_mapping.csv", index=False)

print(f"✓ topic_macro_mapping.csv | {len(mapping)} topics")
mapping

✓ topic_macro_mapping.csv | 30 topics


,topic_id,topic_name,macro_id,macro_name,count
0,-1,graphical interfaces • automatic presentation ...,-1,Other,27
2,1,dimensionality reduction • dimensional dataset...,0,dimensionality reduction • dimensional dataset...,249
8,7,simulation ensembles • ensembles • ensemble • ...,0,dimensionality reduction • dimensional dataset...,111
23,22,diffusion tensor mri • tensor mri • diffusion ...,0,dimensionality reduction • dimensional dataset...,56
4,3,node link diagrams • link diagrams • graph lay...,1,node link diagrams • link diagrams • graph lay...,218
9,8,topic modeling • exploratory search • informat...,1,node link diagrams • link diagrams • graph lay...,150
20,19,sports • sport • contextualized • actionable i...,1,node link diagrams • link diagrams • graph lay...,80
19,18,ray tracing • based rendering • rendering volu...,1,node link diagrams • link diagrams • graph lay...,66
1,0,unstructured meshes • unstructured grids • dir...,2,direct volume rendering • infovis • augmented ...,429
3,2,flow field • flow fields • vortices • computat...,2,direct volume rendering • infovis • augmented ...,238


---
## Manual Label Refinement

After reviewing auto-generated labels, you can manually rename topics/macros below.
Run this cell only after you've reviewed the mapping above.

In [21]:
# Manual topic renaming based on current clustering results
TOPIC_NAMES = {
    0: "Unstructured Meshes & Volume Rendering",
    1: "Dimensionality Reduction & High-Dim Data",
    2: "Flow Fields & CFD",
    3: "Node-Link Diagrams & Graph Layouts",
    4: "Narrative Visualization & Storytelling",
    5: "Dataflow & Visual Programming",
    6: "Vascular & Medical CT Imaging",
    7: "Simulation Ensembles & Weather",
    8: "Topic Modeling & Text Analytics",
    9: "Graphical Perception & Uncertainty",
    10: "Geovisualization & Cartography",
    11: "Eye Tracking & Augmented Reality",
    12: "Deep Learning & Explainable ML",
    13: "Computational Topology & Contour Trees",
    14: "Molecular Dynamics & Simulation",
    15: "Biomedical Imaging & Microscopy",
    16: "Color Perception & Colormaps",
    17: "Volume Graphics & Geophysics",
    18: "Ray Tracing & Volume Rendering",
    19: "Sports Analytics & Insights",
    20: "Network Security & Anomaly Detection",
    21: "3D Interaction & Haptics",
    22: "Diffusion Tensor MRI & DTI",
    23: "Social Media & Twitter Analysis",
    24: "Tensor Fields & Topology",
    25: "Bioinformatics & Genomics",
    26: "Multi-Projector & Display Systems",
    27: "Causality & Temporal Patterns",
    28: "Wavelets & Multiresolution"
}

MACRO_NAMES = {
    0: "High-Dimensional Data Analysis",
    1: "Graph Visualization & Text Mining",
    2: "Volume Rendering & Immersive Tech",
    3: "Visual Programming & ML",
    4: "Social & Biomedical Analytics",
    5: "Imaging & Display Technology",
    6: "Causality & Temporal Analysis",
    7: "Perception & Uncertainty Vis",
    8: "Topological Data Analysis",
    9: "Network Security & Anomaltic",
    10: "Geospatial & Seismic Vis",
    11: "Molecular Simulation"
}

# Apply and save renamed mapping
mapping_renamed = mapping.copy()
mapping_renamed["topic_name"] = mapping_renamed["topic_id"].map(TOPIC_NAMES)
mapping_renamed["macro_name"] = mapping_renamed["macro_id"].map(MACRO_NAMES)
mapping_renamed = mapping_renamed.dropna(subset=["topic_name"])
mapping_renamed.to_csv(OUT / "topic_macro_mapping_renamed.csv", index=False)

print(f"✓ topic_macro_mapping_renamed.csv")
mapping_renamed

✓ topic_macro_mapping_renamed.csv


,topic_id,topic_name,macro_id,macro_name,count
2,1,Dimensionality Reduction & High-Dim Data,0,High-Dimensional Data Analysis,249
8,7,Simulation Ensembles & Weather,0,High-Dimensional Data Analysis,111
23,22,Diffusion Tensor MRI & DTI,0,High-Dimensional Data Analysis,56
4,3,Node-Link Diagrams & Graph Layouts,1,Graph Visualization & Text Mining,218
9,8,Topic Modeling & Text Analytics,1,Graph Visualization & Text Mining,150
20,19,Sports Analytics & Insights,1,Graph Visualization & Text Mining,80
19,18,Ray Tracing & Volume Rendering,1,Graph Visualization & Text Mining,66
1,0,Unstructured Meshes & Volume Rendering,2,Volume Rendering & Immersive Tech,429
3,2,Flow Fields & CFD,2,Volume Rendering & Immersive Tech,238
5,4,Narrative Visualization & Storytelling,2,Volume Rendering & Immersive Tech,196
